In [1]:
import pandas as pd
import re  #regular_expression

In [2]:
pan_data = pd.read_excel("D:\Project_DA\Pan Validation.python\PAN Number Validation Dataset.xlsx")
print(pan_data.head(5))
print(f"total records: {len(pan_data)}")

  Pan_Numbers
0  VGLOD3180G
1  PHOXD7232L
2  MGEPH6532A
3  JJCHK4574O
4  XTQIJ2330L
total records: 10000


In [3]:
total_records = len(pan_data)
total_records

10000

# Data Cleaning:

In [4]:
pan_data.dtypes

Pan_Numbers    object
dtype: object

In [5]:
## object need to converted to string > remove leading and trailing spaces , convert to upper_case

In [6]:
pan_data["Pan_Numbers"] = pan_data["Pan_Numbers"].astype('string').str.strip().str.upper()

In [7]:
## checking null and blank values :

In [8]:
print(pan_data[pan_data["Pan_Numbers"] == ""])
print("total blank values =", len(pan_data[pan_data["Pan_Numbers"] == ""]))

     Pan_Numbers
5019            
5020            
total blank values = 2


In [9]:
print(pan_data[pan_data["Pan_Numbers"].isnull()])
print("total null values =", len(pan_data[pan_data["Pan_Numbers"].isnull()]))

     Pan_Numbers
5022        <NA>
5027        <NA>
5033        <NA>
5043        <NA>
5057        <NA>
...          ...
9961        <NA>
9972        <NA>
9986        <NA>
9987        <NA>
9997        <NA>

[965 rows x 1 columns]
total null values = 965


In [10]:
# replacing blank values to null:

In [11]:
pan_data = pan_data.replace({'Pan_Numbers':''}, pd.NA)

In [12]:
# dropping null values:

In [13]:
pan_data = pan_data.dropna(subset='Pan_Numbers')
pan_data.isna().sum()

Pan_Numbers    0
dtype: int64

In [14]:
# Drop Duplicates:

In [15]:
print("Total records =", len(pan_data["Pan_Numbers"]))
print("Total Unique values =", pan_data["Pan_Numbers"].nunique())
print("After removing duplicate values, totaal records =",len(pan_data.drop_duplicates(subset = 'Pan_Numbers', keep='first')))

Total records = 9033
Total Unique values = 9025
After removing duplicate values, totaal records = 9025


In [16]:
pan_data= pan_data.drop_duplicates(subset = 'Pan_Numbers', keep='first')
pan_data.nunique()

Pan_Numbers    9025
dtype: int64

# Data Validation:

In [17]:
# Function for adjacent repitition :

In [18]:
def adjacent_repitition(pan):
    return any(pan[i] == pan[i+1] for i in range(len(pan)-1))
adjacent_repitition("AABCD")

True

In [19]:
# Function for Sequence check:

In [20]:
def check_sequence(pan):
    return all(ord(pan[i+1]) - ord(pan[i]) == 1 for i in range(len(pan)-1))
check_sequence("AXBYCZ")

False

In [21]:
def check_format(pan):
    return re.match(r'^[A-Z]{5}[0-9]{4}[A-Z]$', pan)

In [22]:
print(check_format("BRDTS1482L"))

<re.Match object; span=(0, 10), match='BRDTS1482L'>


In [23]:
def valid_pan_number(pan):
    if len(pan) != 10:
        return False
        
    if not re.match(r'^[A-Z]{5}[0-9]{4}[A-Z]$', pan):
        return False
        
    if adjacent_repitition(pan):
        return False
        
    if check_sequence(pan):
        return False
        
    return True

In [24]:
pan_data["Validation"] = pan_data["Pan_Numbers"].apply(
    lambda x: 'Valid' if valid_pan_number(x) else 'Invalid')
pan_data.head()

,Pan_Numbers,Validation
0,VGLOD3180G,Valid
1,PHOXD7232L,Valid
2,MGEPH6532A,Valid
3,JJCHK4574O,Invalid
4,XTQIJ2330L,Invalid


# Summary:

In [34]:
total_record_processed = total_records
total_valid = (pan_data["Validation"]=="Valid").sum()
total_invalid = (pan_data["Validation"]=="Invalid").sum()
missing_values = total_record_processed - (total_valid + total_invalid)

In [36]:
print("Total Record Processed = ", total_record_processed)
print("Total valid PANs =", total_valid)
print("Total Invalid PANs =", total_invalid)
print("Total missing or incomplete PANs =", missing_values)

Total Record Processed =  10000
Total valid PANs = 3193
Total Invalid PANs = 5832
Total missing or incomplete PANs = 975


In [38]:
pan_data_summary = pd.DataFrame({
    "TOTAL RECORD PROCESSED": [total_record_processed],
    "TOTAL VALID PAN COUNT": [total_valid],
    "TOTAL INVALID PAN COUNT":[total_invalid],
    "TOTAL MISSING OR INCOMPLETE PAN COUNT":[missing_values]
})
pan_data_summary

,TOTAL RECORD PROCESSED,TOTAL VALID PAN COUNT,TOTAL INVALID PAN COUNT,TOTAL MISSING OR INCOMPLETE PAN COUNT
0,10000,3193,5832,975


# Convert to a excel file 

In [41]:
with pd.ExcelWriter("PAN VALIDATION REPORT.xlsx") as writer:
    pan_data.to_excel(writer, sheet_name = "Pan Validation Status", index = False)
    pan_data_summary.to_excel(writer, sheet_name = "Summary", index = False)